# Modélisation 

### Import des modules 

In [34]:
import pandas as pd

#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error 
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor


### Importation du fichier CSV 'ready_to_fe'
#### ready to feature engineering

In [35]:
building_consumption_ready_to_fe = pd.read_csv("data/Processed/building_consumption_ready_to_fe.csv")

In [36]:
building_consumption_ready_to_fe

,Unnamed: 0,Neighborhood,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),ListOfAllPropertyUseTypes,SiteEUI(kBtu/sf),SiteEnergyUse(kBtu),SteamUse(kBtu),Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,FE1_Is_multi_type
0,0,DOWNTOWN,1927,1.0,12,88434,0,88434,Hotel,81.699997,7.226362e+06,2003882.00,3.946027e+06,1.276453e+06,249.98,0
1,1,DOWNTOWN,1996,1.0,11,103566,15064,88502,"Hotel, Parking, Restaurant",94.800003,8.387933e+06,0.00,3.242851e+06,5.145082e+06,295.86,1
2,2,DOWNTOWN,1969,1.0,41,956110,196718,759392,Hotel,96.000000,7.258702e+07,21566554.00,4.952666e+07,1.493800e+06,2089.28,0
3,3,DOWNTOWN,1926,1.0,10,61320,0,61320,Hotel,110.800003,6.794584e+06,2214446.25,2.768924e+06,1.811213e+06,286.43,0
4,4,DOWNTOWN,1980,1.0,18,175580,62000,113580,"Hotel, Parking, Swimming Pool",114.800003,1.417261e+07,0.00,5.368607e+06,8.803998e+06,505.01,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,3371,GREATER DUWAMISH,1990,1.0,1,12294,0,12294,Office,69.099998,8.497457e+05,0.00,5.242709e+05,3.254750e+05,20.94,0
1588,3372,DOWNTOWN,2004,1.0,1,16000,0,16000,Other - Recreation,59.400002,9.502762e+05,0.00,3.965461e+05,5.537300e+05,32.17,0
1589,3373,MAGNOLIA QUEEN ANNE,1974,1.0,1,13157,0,13157,"Fitness Center/Health Club/Gym, Other - Recrea...",438.200012,5.765898e+06,0.00,1.792159e+06,3.973739e+06,223.54,1
1590,3374,GREATER DUWAMISH,1989,1.0,1,14101,0,14101,"Fitness Center/Health Club/Gym, Food Service, ...",51.000000,7.194712e+05,0.00,3.488702e+05,3.706010e+05,22.11,1


### Feature Engineering

A réaliser : Enrichir le jeu de données actuel avec de nouvelles features issues de celles existantes. 

En règle générale : On utilise la méthode .apply() de Pandas pour créer une nouvelle colonne à partir d'une colonne existante. N'hésitez pas à regarder les exemples dans les chapitres de cours donnés en ressource

#### Les Features : 
<table border="1" style="width: 100%; border-collapse: collapse;">
        <thead>
          <tr>
            <th style="padding: 8px; text-align: left;">Code</th>
            <th style="padding: 8px; text-align: left;">Nom de la feature</th>
            <th style="padding: 8px; text-align: left;">Ce que cette feature représente</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td style="padding: 6px;">FE1</td>
            <td style="padding: 6px;">FE1_is_multi_type</td>
            <td style="padding: 6px;">Permet de savoir si un bâtiment est multi usage ou non</td>
          </tr>
          <tr>
            <td style="padding: 6px;">FE2</td>
            <td style="padding: 6px;">FE2_year_in_activity</td>
            <td style="padding: 6px;">Calculer l'age des bâtiments (permet de savoir si un vieux bâtiment a tendance a consommer davantage)</td>
          </tr>
          <tr>
            <td style="padding: 6px;">FE3</td>
            <td style="padding: 6px;">FE3_floor_area</td>
            <td style="padding: 6px;">Permet de connaitre la superficie d'un étage (indication sur la consommation)</td>
          </tr>
        </tbody>
</table>

La feature FE1 a déja été créé durant l'analyse de données

In [38]:
# Code pour la FE2 
# L'année de référence est 2016 au lieu de 2026 car l'étude des bâtiments date de 2016 
building_consumption_ready_to_fe['FE2_year_in_activity'] = 2016 - building_consumption_ready_to_fe['YearBuilt']
building_consumption_ready_to_fe

,Unnamed: 0,Neighborhood,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),ListOfAllPropertyUseTypes,SiteEUI(kBtu/sf),SiteEnergyUse(kBtu),SteamUse(kBtu),Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,FE1_Is_multi_type,FE2_year_in_activity
0,0,DOWNTOWN,1927,1.0,12,88434,0,88434,Hotel,81.699997,7.226362e+06,2003882.00,3.946027e+06,1.276453e+06,249.98,0,89
1,1,DOWNTOWN,1996,1.0,11,103566,15064,88502,"Hotel, Parking, Restaurant",94.800003,8.387933e+06,0.00,3.242851e+06,5.145082e+06,295.86,1,20
2,2,DOWNTOWN,1969,1.0,41,956110,196718,759392,Hotel,96.000000,7.258702e+07,21566554.00,4.952666e+07,1.493800e+06,2089.28,0,47
3,3,DOWNTOWN,1926,1.0,10,61320,0,61320,Hotel,110.800003,6.794584e+06,2214446.25,2.768924e+06,1.811213e+06,286.43,0,90
4,4,DOWNTOWN,1980,1.0,18,175580,62000,113580,"Hotel, Parking, Swimming Pool",114.800003,1.417261e+07,0.00,5.368607e+06,8.803998e+06,505.01,1,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,3371,GREATER DUWAMISH,1990,1.0,1,12294,0,12294,Office,69.099998,8.497457e+05,0.00,5.242709e+05,3.254750e+05,20.94,0,26
1588,3372,DOWNTOWN,2004,1.0,1,16000,0,16000,Other - Recreation,59.400002,9.502762e+05,0.00,3.965461e+05,5.537300e+05,32.17,0,12
1589,3373,MAGNOLIA QUEEN ANNE,1974,1.0,1,13157,0,13157,"Fitness Center/Health Club/Gym, Other - Recrea...",438.200012,5.765898e+06,0.00,1.792159e+06,3.973739e+06,223.54,1,42
1590,3374,GREATER DUWAMISH,1989,1.0,1,14101,0,14101,"Fitness Center/Health Club/Gym, Food Service, ...",51.000000,7.194712e+05,0.00,3.488702e+05,3.706010e+05,22.11,1,27


In [39]:
# Code pour la FE3
building_consumption_ready_to_fe['FE3_floor_area'] = round(building_consumption_ready_to_fe['PropertyGFABuilding(s)'] / building_consumption_ready_to_fe['NumberofFloors'], 0)
building_consumption_ready_to_fe

,Unnamed: 0,Neighborhood,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),ListOfAllPropertyUseTypes,SiteEUI(kBtu/sf),SiteEnergyUse(kBtu),SteamUse(kBtu),Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,FE1_Is_multi_type,FE2_year_in_activity,FE3_floor_area
0,0,DOWNTOWN,1927,1.0,12,88434,0,88434,Hotel,81.699997,7.226362e+06,2003882.00,3.946027e+06,1.276453e+06,249.98,0,89,7370.0
1,1,DOWNTOWN,1996,1.0,11,103566,15064,88502,"Hotel, Parking, Restaurant",94.800003,8.387933e+06,0.00,3.242851e+06,5.145082e+06,295.86,1,20,8046.0
2,2,DOWNTOWN,1969,1.0,41,956110,196718,759392,Hotel,96.000000,7.258702e+07,21566554.00,4.952666e+07,1.493800e+06,2089.28,0,47,18522.0
3,3,DOWNTOWN,1926,1.0,10,61320,0,61320,Hotel,110.800003,6.794584e+06,2214446.25,2.768924e+06,1.811213e+06,286.43,0,90,6132.0
4,4,DOWNTOWN,1980,1.0,18,175580,62000,113580,"Hotel, Parking, Swimming Pool",114.800003,1.417261e+07,0.00,5.368607e+06,8.803998e+06,505.01,1,36,6310.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1587,3371,GREATER DUWAMISH,1990,1.0,1,12294,0,12294,Office,69.099998,8.497457e+05,0.00,5.242709e+05,3.254750e+05,20.94,0,26,12294.0
1588,3372,DOWNTOWN,2004,1.0,1,16000,0,16000,Other - Recreation,59.400002,9.502762e+05,0.00,3.965461e+05,5.537300e+05,32.17,0,12,16000.0
1589,3373,MAGNOLIA QUEEN ANNE,1974,1.0,1,13157,0,13157,"Fitness Center/Health Club/Gym, Other - Recrea...",438.200012,5.765898e+06,0.00,1.792159e+06,3.973739e+06,223.54,1,42,13157.0
1590,3374,GREATER DUWAMISH,1989,1.0,1,14101,0,14101,"Fitness Center/Health Club/Gym, Food Service, ...",51.000000,7.194712e+05,0.00,3.488702e+05,3.706010e+05,22.11,1,27,14101.0


### Préparation des features pour la modélisation

A réaliser :
* Si ce n'est pas déjà fait, supprimer toutes les colonnes peu pertinentes pour la modélisation.
* Tracer la distribution de la cible pour vous familiariser avec l'ordre de grandeur. En cas d'outliers, mettez en place une démarche pour les supprimer.
* Débarrassez-vous des features redondantes en utilisant une matrice de corrélation de Pearson. Pour cela, utiisez la méthode corr() de Pandas, couplé d'un graphique Heatmap de la librairie Seaborn 
* Réalisez différents graphiques pour comprendre le lien entre vos features et la target (boxplots, scatterplots, pairplot si votre nombre de features numériques n'est pas très élevé).
*  Séparez votre jeu de données en un Pandas DataFrame X (ensemble de feautures) et Pandas Series y (votre target).
* Si vous avez des features catégorielles, il faut les encoder pour que votre modèle fonctionne. Les deux méthodes d'encodage à connaitre sont le OneHotEncoder et le LabelEncoder

#### Suppression de la variable ListOfAllPropertyUseTypes qui n'est plus utile

In [40]:
building_consumption_ready_to_fe.drop(columns=['ListOfAllPropertyUseTypes'], inplace=True)

#### Encodage de la variable Neighborhood avec la methode One-hot
L'intérêt de de numériser les catégories pour facilier le ML    
Le fait d'utiliser cette methode est qu'elle crée 1 colonne par catégorie (en supprimant l'originale avec l'argument drop_first).   

In [41]:
building_consumption_ready_to_fe = pd.get_dummies(building_consumption_ready_to_fe, columns=['Neighborhood'], drop_first=True, dtype=int)

#### Avant la normalisation 

Verifier qu'il ne reste aucun NaN dans les colonnes a normaliser.   
Ne pas faire la normalisation sur la cible y.   
Il faut spliter en Train/Test car nous ne normalisons pas les X_test et Y_test.   

In [43]:
colonnes_a_normaliser = ['SteamUse(kBtu)', 'Electricity(kBtu)', 'NaturalGas(kBtu)', 'PropertyGFABuilding(s)', 'PropertyGFAParking', 'PropertyGFATotal']

# création des Variables et Cible
X = building_consumption_ready_to_fe.drop(columns=['SiteEnergyUse(kBtu)'])
y = building_consumption_ready_to_fe['SiteEnergyUse(kBtu)']

# split (jeux de train et jeux de test)
# le ratio choisi est 80% Train et 20% Test
# le random_state est la seed pour les jeux de test (si besoin de reproduire exactement le meme jeu de test utiliser la meme seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


X_train = X_train.copy()
X_test = X_test.copy()

# Création d'on objet de la classe MinMaxScaler
scaler = MinMaxScaler()

#### Normalisation des Features. Dans le but de réduire les amplitudes entre les variables (Facilite le ML).    
* SteamUse(kBtu).   
* Electricity(kBtu).	   
* NaturalGas(kBtu).  
* PropertyGFABuilding(s).   
* PropertyGFAParking.   
* PropertyGFATotal.  

In [44]:
X_train[colonnes_a_normaliser] = scaler.fit_transform(X_train[colonnes_a_normaliser])
X_test[colonnes_a_normaliser] = scaler.fit_transform(X_test[colonnes_a_normaliser])

In [45]:
X_train

,Unnamed: 0,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),SiteEUI(kBtu/sf),SteamUse(kBtu),Electricity(kBtu),...,Neighborhood_DOWNTOWN,Neighborhood_EAST,Neighborhood_GREATER DUWAMISH,Neighborhood_LAKE UNION,Neighborhood_MAGNOLIA QUEEN ANNE,Neighborhood_NORTH,Neighborhood_NORTHEAST,Neighborhood_NORTHWEST,Neighborhood_SOUTHEAST,Neighborhood_SOUTHWEST
1545,3235,2014,1.0,7,0.147314,0.286158,0.083498,23.900000,0.0,0.015878,...,0,0,1,0,0,0,0,0,0,0
1509,3125,1904,1.0,4,0.028132,0.000000,0.031517,20.600000,0.0,0.005302,...,0,0,1,0,0,0,0,0,0,0
744,1046,1961,1.0,1,0.012208,0.000000,0.015648,19.799999,0.0,0.002305,...,0,0,1,0,0,0,0,0,0,0
462,491,1926,1.0,6,0.039491,0.000000,0.042836,33.700001,0.0,0.009026,...,1,0,0,0,0,0,0,0,0,0
1356,2623,1961,1.0,2,0.020418,0.000000,0.023829,109.599998,0.0,0.009681,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1130,2003,1966,1.0,1,0.005971,0.000000,0.009433,30.200001,0.0,0.001720,...,0,0,1,0,0,0,0,0,0,0
1294,2404,1900,1.0,5,0.012378,0.000000,0.015817,65.000000,0.0,0.007498,...,1,0,0,0,0,0,0,0,0,0
860,1278,1998,1.0,1,0.008794,0.000000,0.012246,206.199997,0.0,0.009583,...,0,0,0,0,1,0,0,0,0,0
1459,2993,1950,1.0,1,0.045008,0.000000,0.048334,30.000000,0.0,0.004716,...,0,0,0,0,0,0,0,0,0,0


### Comparaison de différents modèles supervisés

A réaliser :
* Pour chaque algorithme que vous allez tester, vous devez :
    * Réaliser au préalable une séparation en jeu d'apprentissage et jeu de test via une validation croisée.
    * Si les features quantitatives que vous souhaitez utiliser ont des ordres de grandeur très différents les uns des autres, et que vous utilisez un algorithme de regression qui est sensible à cette différence, alors il faut réaliser un scaling (normalisation) de la donnée au préalable.
    * Entrainer le modèle sur le jeu de Train
    * Prédire la cible sur la donnée de test (nous appelons cette étape, l'inférence).
    * Calculer les métriques de performance R2, MAE et RMSE sur le jeu de train et de test.
    * Interpréter les résultats pour juger de la fiabilité de l'algorithme.
* Vous pouvez choisir par exemple de tester un modèle linéaire, un modèle à base d'arbres et un modèle de type SVM
* Déterminer le modèle le plus performant parmi ceux testés.

In [ ]:
# CODE COMPARAISON DES MODELES

### Optimisation et interprétation du modèle

A réaliser :
* Reprennez le meilleur algorithme que vous avez sécurisé via l'étape précédente, et réalisez une GridSearch de petite taille sur au moins 3 hyperparamètres.
* Si le meilleur modèle fait partie de la famille des modèles à arbres (RandomForest, GradientBoosting) alors utilisez la fonctionnalité feature importance pour identifier les features les plus impactantes sur la performance du modèle. Sinon, utilisez la méthode Permutation Importance de sklearn.

In [ ]:
# CODE OPTIMISATION ET INTERPRETATION DU MODELE